# Day 06 — Document Loaders & Text Splitting (hands-on)

Companion notebook to [`../notes.md`](../notes.md). Loads a small sample document, then compares
**fixed-size** splitting against **recursive** splitting, and shows exactly what chunk overlap looks
like in real output.

No API key needed — everything here runs locally.

In [1]:
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

DATA_PATH = Path("data/sample_article.txt")

C:\Users\ramu\AppData\Local\Temp\ipykernel_20980\1896677247.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


C:\Users\ramu\OneDrive\Desktop\Agentic-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the document

A document loader turns a raw file into LangChain `Document` objects — each with `page_content` (the
text) and `metadata` (like where it came from). This is the same shape no matter what loader you use
(PDF, DOCX, webpage) — see `notes.md` Section 1.

In [2]:
loader = TextLoader(str(DATA_PATH), encoding="utf-8")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print("Metadata:", docs[0].metadata)
print("Total characters:", len(docs[0].page_content))
print()
print(docs[0].page_content)

Loaded 1 document(s)
Metadata: {'source': 'data\\sample_article.txt'}
Total characters: 886

Paris is the capital of France. It is known for the Eiffel Tower, which was completed in 1889 and stands about 330 meters tall. Millions of tourists visit every year to see its museums, architecture, and cafes along the Seine.

The Louvre Museum, located in the heart of Paris, is the world's largest art museum. It houses the Mona Lisa, painted by Leonardo da Vinci, along with thousands of other works spanning centuries of human history.

France has a temperate climate with warm summers and mild winters. Agriculture remains important to the French economy, particularly wine production in regions like Bordeaux and Burgundy, which are famous worldwide for their vineyards.

Paris is also a major center for fashion and cuisine. Many of the world's leading fashion houses were founded there, and French cooking techniques form the basis of much of modern Western culinary training.



## 2. Fixed-size splitting — cuts wherever the character count lands

`CharacterTextSplitter` with an empty separator just cuts every N characters, no matter what's there.
Watch the start/end of these chunks — they cut mid-word.

In [3]:
fixed_splitter = CharacterTextSplitter(separator="", chunk_size=120, chunk_overlap=0)
fixed_chunks = fixed_splitter.split_text(docs[0].page_content)

print(f"{len(fixed_chunks)} chunks\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

8 chunks

--- chunk 0 (120 chars) ---
Paris is the capital of France. It is known for the Eiffel Tower, which was completed in 1889 and stands about 330 meter

--- chunk 1 (120 chars) ---
s tall. Millions of tourists visit every year to see its museums, architecture, and cafes along the Seine.

The Louvre M

--- chunk 2 (120 chars) ---
useum, located in the heart of Paris, is the world's largest art museum. It houses the Mona Lisa, painted by Leonardo da

--- chunk 3 (119 chars) ---
Vinci, along with thousands of other works spanning centuries of human history.

France has a temperate climate with wa

--- chunk 4 (120 chars) ---
rm summers and mild winters. Agriculture remains important to the French economy, particularly wine production in region

--- chunk 5 (119 chars) ---
s like Bordeaux and Burgundy, which are famous worldwide for their vineyards.

Paris is also a major center for fashion

--- chunk 6 (120 chars) ---
and cuisine. Many of the world's leading fashion houses were fou

Notice chunk 0 above ends mid-word (something like `...stands about 330 meter`) and the next chunk
picks up wherever the character count happened to land — not at a word or sentence boundary. This is
exactly the "cuts mid-word" problem from `notes.md` Section 2.

## 3. Recursive splitting — tries natural boundaries first

`RecursiveCharacterTextSplitter` tries paragraph breaks, then sentence breaks, then word breaks, before
resorting to a hard cut. Same source text, same `chunk_size`, much more readable chunks.

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
recursive_chunks = recursive_splitter.split_text(docs[0].page_content)

print(f"{len(recursive_chunks)} chunks\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

10 chunks

--- chunk 0 (114 chars) ---
Paris is the capital of France. It is known for the Eiffel Tower, which was completed in 1889 and stands about 330

--- chunk 1 (117 chars) ---
stands about 330 meters tall. Millions of tourists visit every year to see its museums, architecture, and cafes along

--- chunk 2 (26 chars) ---
and cafes along the Seine.

--- chunk 3 (117 chars) ---
The Louvre Museum, located in the heart of Paris, is the world's largest art museum. It houses the Mona Lisa, painted

--- chunk 4 (113 chars) ---
Mona Lisa, painted by Leonardo da Vinci, along with thousands of other works spanning centuries of human history.

--- chunk 5 (119 chars) ---
France has a temperate climate with warm summers and mild winters. Agriculture remains important to the French economy,

--- chunk 6 (118 chars) ---
the French economy, particularly wine production in regions like Bordeaux and Burgundy, which are famous worldwide for

--- chunk 7 (30 chars) ---
worldwide for their vineyards

## 4. Seeing chunk overlap in real output

`chunk_overlap=20` means each chunk repeats a little of the previous chunk's ending. Let's print the
tail of chunk 0 next to the head of chunk 1 to see the shared text directly.

In [5]:
print("End of chunk 0:  ", repr(recursive_chunks[0][-35:]))
print("Start of chunk 1: ", repr(recursive_chunks[1][:35]))
print()
print("That overlap is exactly what protects a sentence sitting right at a chunk boundary")
print("from being lost — it still appears whole in at least one of the two chunks.")

End of chunk 0:   'pleted in 1889 and stands about 330'
Start of chunk 1:  'stands about 330 meters tall. Milli'

That overlap is exactly what protects a sentence sitting right at a chunk boundary
from being lost — it still appears whole in at least one of the two chunks.


## 5. A cleaner view: chunk size matched to paragraph length

If `chunk_size` comfortably fits a whole paragraph, recursive splitting aligns almost perfectly to
paragraph boundaries — no mid-sentence cuts at all.

In [6]:
clean_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
clean_chunks = clean_splitter.split_text(docs[0].page_content)

print(f"{len(clean_chunks)} chunks\n")
for i, chunk in enumerate(clean_chunks):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

4 chunks

--- chunk 0 (226 chars) ---
Paris is the capital of France. It is known for the Eiffel Tower, which was completed in 1889 and stands about 330 meters tall. Millions of tourists visit every year to see its museums, architecture, and cafes along the Seine.

--- chunk 1 (212 chars) ---
The Louvre Museum, located in the heart of Paris, is the world's largest art museum. It houses the Mona Lisa, painted by Leonardo da Vinci, along with thousands of other works spanning centuries of human history.

--- chunk 2 (235 chars) ---
France has a temperate climate with warm summers and mild winters. Agriculture remains important to the French economy, particularly wine production in regions like Bordeaux and Burgundy, which are famous worldwide for their vineyards.

--- chunk 3 (206 chars) ---
Paris is also a major center for fashion and cuisine. Many of the world's leading fashion houses were founded there, and French cooking techniques form the basis of much of modern Western culinary tr

## Try it yourself

- Change `chunk_size` and `chunk_overlap` above and re-run — watch how the chunk count and boundaries
  change.
- Swap in your own `.txt` file in `data/` and re-run Section 1 onward.
- If you have a PDF handy, try `PyPDFLoader` or `PyMuPDFLoader` (both installed) in place of
  `TextLoader`, and compare the `metadata` you get back (page numbers show up here).